# Reverse a string with a tiny encoder-decoder

T5 reads one sequence and writes another. Here you build that shape in NumPy: span corruption
from the lesson, the teacher-forced decoder input, cross-attention from the decoder into the
encoder, the loss, and the gradients for the two pieces that learn. The task is to reverse
strings of four letters. After a hundred gradient steps the cross-attention weights line up
along the anti-diagonal, and greedy decoding reverses strings the model has never seen.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

vocab = ["<pad>", "<s>", "<e>", "a", "b", "c", "d"]
idx = {w: i for i, w in enumerate(vocab)}
PAD, START, END = 0, 1, 2
n_vocab = len(vocab)
L = 4                        # letters per source string
T_in, T_out = L, L + 1       # the target is the L letters reversed, then <e>
d = 16                       # width of every vector inside the model


def to_ids(text):
    return np.array([idx[ch] for ch in text])


def to_text(ids):
    return "".join(vocab[i] for i in ids)


def random_pair(rng):
    """A source string and its target: the letters reversed, then <e>."""
    source = "".join(rng.choice(list("abcd"), size=L))
    return to_ids(source), np.append(to_ids(source[::-1]), END)


train_pairs = [random_pair(rng) for _ in range(32)]
seen = {to_text(src) for src, _ in train_pairs}
test_pairs = []
while len(test_pairs) < 16:
    src, tgt = random_pair(rng)
    if to_text(src) not in seen:
        test_pairs.append((src, tgt))
        seen.add(to_text(src))


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


# The encoder: token and position embeddings, then one layer of attention in both directions.
E_enc = 0.5 * rng.normal(size=(n_vocab, d))
P_enc = 0.5 * rng.normal(size=(T_in, d))
Wq_enc, Wk_enc, Wv_enc = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))


def encode_source(src):
    """Encoder output, one row per source token. Shape (T_in, d)."""
    X = E_enc[src] + P_enc[np.arange(len(src))]
    Q, K, V = X @ Wq_enc, X @ Wk_enc, X @ Wv_enc
    weights = softmax(Q @ K.T / np.sqrt(d), axis=1)
    return X + weights @ V


# The decoder: its own embeddings and one causal self-attention layer. Cross-attention (Part 2)
# and the output layer come after it.
E_dec = 0.5 * rng.normal(size=(n_vocab, d))
P_dec = 0.5 * rng.normal(size=(T_out, d))
Wq_dec, Wk_dec, Wv_dec = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))


def decoder_self_attention(dec_in):
    """Decoder states after causal self-attention over the decoder input ids. Shape (len(dec_in), d)."""
    X = E_dec[dec_in] + P_dec[np.arange(len(dec_in))]
    Q, K, V = X @ Wq_dec, X @ Wk_dec, X @ Wv_dec
    scores = Q @ K.T / np.sqrt(d)
    scores = np.where(np.tril(np.ones(scores.shape, dtype=bool)), scores, -np.inf)
    return X + softmax(scores, axis=1) @ V


# Cross-attention weights, a learned bias B with one number per (decoder position, source
# position) pair that is added to the cross-attention scores, and the output layer.
# The encoder and the decoder self-attention stay random. B, W_out and b_out learn.
Wq_x, Wk_x, Wv_x = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))
B = np.zeros((T_out, T_in))
W_out = rng.normal(size=(d, n_vocab)) / np.sqrt(d)
b_out = np.zeros(n_vocab)

src, tgt = train_pairs[0]
print("source", to_text(src), "-> target", to_text(tgt))
print(len(train_pairs), "training pairs,", len(test_pairs), "test pairs the model never trains on")

## Part 1 (coding task): Span corruption

T5 is pretrained by hiding runs of consecutive words, called spans, and asking the model
to write them out. In the input, each hidden span is replaced by one sentinel token,
`<X0>`, `<X1>` and so on. The target lists each sentinel followed by the words it hid, and
ends with one more sentinel that marks the end. The target is much shorter than the
sentence, which makes pretraining cheaper.

Write `span_corrupt(words, spans)`. `words` is a list of strings and `spans` is a sorted
list of `(start, end)` pairs to hide, with `end` exclusive, as in Python slices. Return
`(inputs, targets)` as two lists of strings. The lesson's example hides "for inviting" and
"last" in "Thank you for inviting me to your party last week .".

In [ ]:
def span_corrupt(words, spans):
    """(inputs, targets) for T5's span corruption. spans: sorted (start, end) pairs, end exclusive."""
    # TODO
    raise NotImplementedError

In [ ]:
words = "Thank you for inviting me to your party last week .".split()
inp, tgt_words = span_corrupt(words, [(2, 4), (8, 9)])
assert inp == "Thank you <X0> me to your party <X1> week .".split(), f"input: got {' '.join(inp)}"
assert tgt_words == "<X0> for inviting <X1> last <X2>".split(), f"target: got {' '.join(tgt_words)}"
assert len(inp) == 10 and len(tgt_words) == 6
inp2, tgt2 = span_corrupt(words, [])
assert inp2 == words and tgt2 == ["<X0>"], "with nothing hidden the input is the sentence and the target is one sentinel"
inp3, tgt3 = span_corrupt(words, [(0, 2), (10, 11)])
assert inp3 == "<X0> for inviting me to your party last week <X1>".split(), "a span may start at the first word or end at the last"
assert tgt3 == "<X0> Thank you <X1> . <X2>".split()
print("input: ", " ".join(inp))
print("target:", " ".join(tgt_words))
print("Part 1 passed")

## Part 2 (coding task): Cross-attention

The decoder needs a way to look at the input. In cross-attention the queries come from
the decoder states and the keys and values come from the encoder output. The scores form
a grid with one row per decoder position and one column per source token, `T_out x T_in`,
and a softmax along each row turns them into weights. No causal mask is needed, because
the whole input is known before the decoder starts. Our scores get one extra term,
`B[t, s]`, a learned number for each pair of positions. T5's attention adds a learned
position bias to its scores in a similar way. `B` starts at zero and is what Part 5
trains, together with the output layer.

Write `cross_attention(H_dec, H_enc)`. Compute `Q = H_dec @ Wq_x`, `K = H_enc @ Wk_x`,
`V = H_enc @ Wv_x`, then `scores = Q @ K.T / np.sqrt(d) + B[:len(H_dec)]`, the row
softmax, and return `(weights @ V, weights)`.

In [ ]:
def cross_attention(H_dec, H_enc):
    """Queries from the decoder states, keys and values from the encoder output.
    Returns (out, weights): out is one d-vector per decoder position, weights is len(H_dec) x T_in."""
    # TODO
    raise NotImplementedError

In [ ]:
src, tgt = train_pairs[0]
H_enc = encode_source(src)
dec_in = np.append(START, tgt[:-1])
H_dec = decoder_self_attention(dec_in)
out, A = cross_attention(H_dec, H_enc)
assert out.shape == (T_out, d), f"out should be {(T_out, d)}, got {out.shape}"
assert A.shape == (T_out, T_in), f"weights should be {(T_out, T_in)}, one row per decoder position, got {A.shape}"
assert np.allclose(A.sum(axis=1), 1), "each row of weights should sum to 1"
assert (A > 0).all(), "no mask here: every decoder position may look at every source token"
Q, K, V = H_dec @ Wq_x, H_enc @ Wk_x, H_enc @ Wv_x
A_ref = softmax(Q @ K.T / np.sqrt(d) + B, axis=1)
assert np.allclose(A, A_ref), "weights should be the row softmax of Q K^T / sqrt(d) + B"
assert np.allclose(out, A_ref @ V), "out should be weights @ V"
# The bias steers the weights: a large B[0, 3] sends row 0 to source position 3.
B_saved = B.copy()
B[0, 3] = 10.0
_, A_steered = cross_attention(H_dec, H_enc)
assert A_steered[0, 3] > 0.99, "B[0, 3] = 10 should put almost all of row 0 on source position 3"
assert np.allclose(A_steered[1:], A[1:]), "the other rows should not change"
B[:] = B_saved
# A shorter decoder prefix uses the first rows of B.
out_prefix, A_prefix = cross_attention(H_dec[:2], H_enc)
assert A_prefix.shape == (2, T_in) and np.allclose(A_prefix, A[:2]), "a prefix of 2 decoder positions gives the first 2 rows"
print("cross-attention weights for source", to_text(src), "(rows: decoder positions, columns: source positions)")
print(np.round(A, 3))
print("Part 2 passed")

## Part 3 (coding task): Teacher forcing

During training the target is known, so the decoder reads the whole target at once,
shifted right: `<s>` in front and the last token dropped. At each position it predicts
the next target token. This is teacher forcing: the decoder sees the true previous tokens,
never its own guesses. The decoder state at a position is its self-attention output plus
what cross-attention gathered, and the output layer turns that into one score per
vocabulary entry. Feed the decoder the unshifted target and each position can see the
token it must predict, so the model learns to copy and is useless at test time.

Write `teacher_forcing(src, tgt)`. Build the decoder input from `tgt`, run
`encode_source`, `decoder_self_attention` and `cross_attention`, and return
`(logits, weights)` where `logits = (H_dec + out) @ W_out + b_out` has shape
`T_out x n_vocab`.

In [ ]:
def teacher_forcing(src, tgt):
    """Run the encoder on src and the decoder on the shifted target.
    Returns (logits, weights): logits is T_out x n_vocab, weights is the cross-attention matrix."""
    # TODO
    raise NotImplementedError

In [ ]:
src, tgt = train_pairs[0]
logits, A = teacher_forcing(src, tgt)
assert logits.shape == (T_out, n_vocab), f"logits should be {(T_out, n_vocab)}, got {logits.shape}"
assert A.shape == (T_out, T_in)
H_enc = encode_source(src)
H_dec = decoder_self_attention(np.append(START, tgt[:-1]))
out_ref, A_ref = cross_attention(H_dec, H_enc)
assert np.allclose(logits, (H_dec + out_ref) @ W_out + b_out), "logits do not match: the decoder input is <s> followed by all but the last target token"
assert np.allclose(A, A_ref)
# Feeding the target itself, unshifted, gives different logits. That is the copy trap.
H_unshifted = decoder_self_attention(tgt)
assert not np.allclose(logits, (H_unshifted + cross_attention(H_unshifted, H_enc)[0]) @ W_out + b_out), "the decoder must not read the unshifted target"
print("source       ", to_text(src))
print("decoder input", to_text(np.append(START, tgt[:-1])))
print("target       ", to_text(tgt))
print("Part 3 passed")

## Part 4 (coding task): The loss and its gradients

The loss is the mean cross-entropy of the `T_out` rows of logits against the target ids,
as in GPT. Two things learn here: the output layer and the bias `B`. For a softmax
classifier, the gradient of the loss with respect to the logits at position `t` is
`softmax(logits[t]) - onehot(tgt[t])`, divided by `T_out`. Stack those rows as `G`. Then:

- Output layer: `dW_out = (H_dec + out).T @ G` and `db_out = G.sum(axis=0)`.
- Back to the cross-attention output: `D = G @ W_out.T`, because `out` enters the logits
  through the sum `H_dec + out`.
- Back through `out = weights @ V`: the gradient with respect to `weights` is
  `dA = D @ V.T`, since `weights[t, s]` multiplies `V[s]` into row `t`.
- Back through the row softmax: `dScores = weights * (dA - (weights * dA).sum(axis=1, keepdims=True))`.
- `B` is added straight to the scores, so `dB = dScores`.

Write `loss_and_gradients(src, tgt)`. It returns `(loss, dB, dW_out, db_out)`. You need
the pieces inside `teacher_forcing`, so compute them again here instead of calling it.
The check compares every gradient with finite differences.

In [ ]:
def loss_and_gradients(src, tgt):
    """Teacher-forced loss on one pair and its gradients with respect to B, W_out and b_out."""
    # TODO
    raise NotImplementedError

In [ ]:
src, tgt = train_pairs[0]
loss, dB, dW_out, db_out = loss_and_gradients(src, tgt)
logits, _ = teacher_forcing(src, tgt)
p = softmax(logits, axis=1)
assert abs(loss - (-np.log(p[np.arange(T_out), tgt]).mean())) < 1e-9, "the loss is the mean cross-entropy over the T_out positions"
assert dB.shape == B.shape and dW_out.shape == W_out.shape and db_out.shape == b_out.shape, "the gradients have the shapes of B, W_out and b_out"

def loss_only(src, tgt):
    return loss_and_gradients(src, tgt)[0]

eps = 1e-6
for t, s in [(0, 3), (2, 1), (4, 0)]:
    B[t, s] += eps
    up = loss_only(src, tgt)
    B[t, s] -= 2 * eps
    down = loss_only(src, tgt)
    B[t, s] += eps
    numeric = (up - down) / (2 * eps)
    assert abs(numeric - dB[t, s]) < 1e-6, f"dB[{t}, {s}] should be {numeric:.6f}, got {dB[t, s]:.6f}"
for i, j in [(0, 3), (9, 5)]:
    W_out[i, j] += eps
    up = loss_only(src, tgt)
    W_out[i, j] -= 2 * eps
    down = loss_only(src, tgt)
    W_out[i, j] += eps
    numeric = (up - down) / (2 * eps)
    assert abs(numeric - dW_out[i, j]) < 1e-6, f"dW_out[{i}, {j}] should be {numeric:.6f}, got {dW_out[i, j]:.6f}"
b_out[2] += eps
up = loss_only(src, tgt)
b_out[2] -= 2 * eps
down = loss_only(src, tgt)
b_out[2] += eps
numeric = (up - down) / (2 * eps)
assert abs(numeric - db_out[2]) < 1e-6, f"db_out[2] should be {numeric:.6f}, got {db_out[2]:.6f}"
print(f"loss on the first pair before training: {loss:.3f} (ln {n_vocab} = {np.log(n_vocab):.3f})")
print("Part 4 passed")

## Part 5 (coding task): Train

Now gradient descent. Each step averages the gradients over all the training pairs and
moves `B`, `W_out` and `b_out` against them. A reversal needs decoder position `t` to look
at source position `L - 1 - t`, and nothing in the model knows that yet. Watch the
cross-attention weights before and after.

Write `train(pairs, steps, lr)`. For each step, average `loss_and_gradients` over the
pairs, update the three arrays, and record the mean loss. Change the arrays in place with
`B[:] -= lr * dB` and the same for the other two. Do not write `B = B - lr * dB` inside the function: that builds a new array and binds the local name `B` to it, so the arrays the other functions read never change. Return
the list of mean losses, one per step.

In [ ]:
def train(pairs, steps, lr):
    """Gradient descent on B, W_out and b_out, in place. Returns the mean loss at each step."""
    # TODO
    raise NotImplementedError

In [ ]:
src, tgt = test_pairs[0]
_, A_before = teacher_forcing(src, tgt)
B_start, W_start = B.copy(), W_out.copy()
losses = train(train_pairs, 100, 2.0)
assert len(losses) == 100, "one mean loss per step"
assert not np.allclose(B, B_start) and not np.allclose(W_out, W_start), "train must change B and W_out in place"
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
assert losses[-1] < 0.3, f"after 100 steps the loss should be under 0.3, got {losses[-1]:.3f}"
_, A_after = teacher_forcing(src, tgt)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].plot(losses)
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
for ax, weights, title in [(axes[1], A_before, "cross-attention before"), (axes[2], A_after, "cross-attention after")]:
    ax.imshow(weights, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("source position")
    ax.set_ylabel("decoder position")
plt.tight_layout()
plt.show()
print("source position each decoder position looks at after training:", A_after.argmax(axis=1))
assert (A_after[:L].argmax(axis=1) == np.arange(L)[::-1]).all(), "decoder position t should look at source position L - 1 - t"
print("Part 5 passed")

## Part 6 (coding task): Greedy decoding

At test time there is no target. The encoder runs once. The decoder starts from `<s>`,
predicts the next token, appends it, and repeats until it writes `<e>` or runs out of
positions. Each step runs the decoder on the whole prefix so far and reads the logits at
the last position.

Write `greedy_decode(src)`. Return the array of ids the model wrote, without the leading
`<s>`, stopping after `<e>` or after `T_out` tokens.

In [ ]:
def greedy_decode(src):
    """Reverse src with the trained model, one most-likely token at a time. Returns the ids it wrote."""
    # TODO
    raise NotImplementedError

In [ ]:
correct = 0
for src, tgt in test_pairs:
    written = greedy_decode(src)
    assert len(written) <= T_out, "at most T_out tokens"
    ok = len(written) == len(tgt) and (written == tgt).all()
    correct += ok
    note = "" if ok else "(expected " + to_text(tgt) + ")"
    print(f"{to_text(src)} -> {to_text(written):12s} {note}")
print(f"{correct} of {len(test_pairs)} unseen strings reversed exactly")
assert correct >= 14, "the trained model should reverse nearly every unseen string"
# Each written token is the model's most likely choice given what it wrote before.
src, tgt = test_pairs[0]
written = greedy_decode(src)
H_enc = encode_source(src)
for t in range(len(written)):
    H_dec = decoder_self_attention(np.append(START, written[:t]))
    out, _ = cross_attention(H_dec, H_enc)
    assert written[t] == np.argmax((H_dec[-1] + out[-1]) @ W_out + b_out), f"token {t} is not the most likely choice"
print("Part 6 passed")

## Solution

### Part 1: Span corruption

In [ ]:
def span_corrupt(words, spans):
    """(inputs, targets) for T5's span corruption. spans: sorted (start, end) pairs, end exclusive."""
    inputs, targets, pos = [], [], 0
    for k, (a, b) in enumerate(spans):
        inputs += words[pos:a] + [f"<X{k}>"]
        targets += [f"<X{k}>"] + words[a:b]
        pos = b
    inputs += words[pos:]
    targets += [f"<X{len(spans)}>"]
    return inputs, targets

### Part 2: Cross-attention

In [ ]:
def cross_attention(H_dec, H_enc):
    """Queries from the decoder states, keys and values from the encoder output.
    Returns (out, weights): out is one d-vector per decoder position, weights is len(H_dec) x T_in."""
    Q, K, V = H_dec @ Wq_x, H_enc @ Wk_x, H_enc @ Wv_x
    scores = Q @ K.T / np.sqrt(d) + B[:len(H_dec)]
    weights = softmax(scores, axis=1)
    return weights @ V, weights

### Part 3: Teacher forcing

In [ ]:
def teacher_forcing(src, tgt):
    """Run the encoder on src and the decoder on the shifted target.
    Returns (logits, weights): logits is T_out x n_vocab, weights is the cross-attention matrix."""
    H_enc = encode_source(src)
    dec_in = np.append(START, tgt[:-1])
    H_dec = decoder_self_attention(dec_in)
    out, weights = cross_attention(H_dec, H_enc)
    return (H_dec + out) @ W_out + b_out, weights

### Part 4: The loss and its gradients

In [ ]:
def loss_and_gradients(src, tgt):
    """Teacher-forced loss on one pair and its gradients with respect to B, W_out and b_out."""
    H_enc = encode_source(src)
    H_dec = decoder_self_attention(np.append(START, tgt[:-1]))
    V = H_enc @ Wv_x
    out, weights = cross_attention(H_dec, H_enc)
    H = H_dec + out
    probs = softmax(H @ W_out + b_out, axis=1)
    loss = -np.log(probs[np.arange(T_out), tgt]).mean()
    G = probs.copy()
    G[np.arange(T_out), tgt] -= 1
    G /= T_out
    dW_out, db_out = H.T @ G, G.sum(axis=0)
    D = G @ W_out.T
    dA = D @ V.T
    dB = weights * (dA - (weights * dA).sum(axis=1, keepdims=True))
    return loss, dB, dW_out, db_out

### Part 5: Train

In [ ]:
def train(pairs, steps, lr):
    """Gradient descent on B, W_out and b_out, in place. Returns the mean loss at each step."""
    losses = []
    for step in range(steps):
        results = [loss_and_gradients(src, tgt) for src, tgt in pairs]
        losses.append(np.mean([r[0] for r in results]))
        B[:] -= lr * np.mean([r[1] for r in results], axis=0)
        W_out[:] -= lr * np.mean([r[2] for r in results], axis=0)
        b_out[:] -= lr * np.mean([r[3] for r in results], axis=0)
    return losses

### Part 6: Greedy decoding

In [ ]:
def greedy_decode(src):
    """Reverse src with the trained model, one most-likely token at a time. Returns the ids it wrote."""
    H_enc = encode_source(src)
    dec_in = [START]
    while len(dec_in) <= T_out:
        H_dec = decoder_self_attention(np.array(dec_in))
        out, _ = cross_attention(H_dec, H_enc)
        logits = (H_dec[-1] + out[-1]) @ W_out + b_out
        dec_in.append(int(np.argmax(logits)))
        if dec_in[-1] == END:
            break
    return np.array(dec_in[1:])